# Stellar-Age BNN (RGB): read → normalize → train → calibrate

End-to-end pipeline for the APOKASC **RGB** catalog (`apokacs_rgb_bnn.csv`):

1. **Read** the single catalog, dedup by `APOGEE_ID` (safety net — this catalog has
   no duplicates today), and make a **stratified** train/val/test split on logAge
   bins, since the age tails hold single-digit star counts.
2. **Normalize** — drop unphysical (>13.8 Gyr) ages, derive `[C/N]`, and standardize
   features using **train-only** statistics (no leakage). Reuses the helpers in
   `prepare_dataset.py`.
3. **Train** the robust BNN with input-uncertainty propagation and minibatch-correct
   ELBO scaling (applied inside `train_smooth_bnn`).
4. **Calibrate** the predicted uncertainties on the val split, then **evaluate once**
   on the held-out test split, including ridge/kNN baseline comparisons.

Designed to run on Gadi (or anywhere the repo + dataset live).

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# The repo holding prepare_dataset.py / train_bnn.py is located in the next cell.

: 

In [2]:

CANDIDATES = [
    '~/scr_mk27/bingo-modern',
    '~/code/bingo-modern',
    '.',
]
repo = next((p for p in map(os.path.expanduser, CANDIDATES)
             if os.path.isfile(os.path.join(p, 'prepare_dataset.py'))), None)
if repo is None:
    raise FileNotFoundError('Could not find prepare_dataset.py. Add its dir to CANDIDATES.')
repo = os.path.abspath(repo)
os.chdir(repo)
sys.path.insert(0, repo)
print('repo:', repo)

import prepare_dataset as prep
from train_bnn import (
    set_seed,
    BayesianNeuralNetwork,
    train_smooth_bnn,
    get_targeted_posterior_samples,
    analyze_targeted_results,
    device,
)
print(f'Using device: {device}')

## 2. Configuration

In [ ]:
# APOKASC RGB catalog in BNN format (single, unsplit file).
DATASET = '~/scr_mk27/bulge-ages-and-orbits/data/apokacs_rgb_bnn.csv'
OUTPUT_DIR = Path('./BNN_targeted_output_RGB')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
TEST_FRAC = 0.20         # held-out fraction, touched exactly once (section 7)
VAL_FRAC = 0.15          # tuning + uncertainty-calibration split (never trained on)
SEED = 42
NUM_ITERATIONS = 20000
BATCH_SIZE = 32
HIDDEN_DIM = 32
INITIAL_LR = 0.005
LR_DECAY_PER_EPOCH = 0.995  # ~0.5x LR over ~140 epochs; set 1.0 for constant LR
WEIGHT_CLIP = 3.0        # max inverse-frequency weight ratio; the sparsest age bins
                         # hold ~8 stars vs ~1.2k in the peak (unclipped ratio ~150x),
                         # so without a clip a handful of stars dominates the gradient

# Features the model sees (base columns + the engineered [C/N]).
FEATURES = prep.BASE_FEATURES + ['C_N']
FEATURE_COLS = [f'{f}_NORM' for f in FEATURES]
ERROR_COLS = [f'{f}_ERR_NORM' for f in FEATURES]
FEATURES

## 3. Read & split the dataset

Dedup by `APOGEE_ID`, keeping the highest-SNR row. This catalog currently contains
**no** duplicates (5660 → 5660), but the dedup stays as a safety net so a future
re-export with repeat observations cannot leak the same star across splits.

The split is **stratified on logAge quantile bins** into train/val/test: the young
and old tails hold single-digit star counts, so a plain random split can leave
val/test with zero tail stars and make tail metrics meaningless. Val is used for
tuning and uncertainty calibration; **test is touched exactly once**, in section 7.

In [ ]:
df = pd.read_csv(DATASET)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]   # tolerate a stray index column
df['age'] = 10.0 ** df[prep.TARGET]   # Gyr, for the unphysical-age cut in clean_labels
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')

# Safety net: keep the highest-SNR row per star (no-op on the current catalog).
before = len(df)
df = (df.sort_values('SNR', ascending=False)
        .drop_duplicates(prep.ID_COL, keep='first')
        .reset_index(drop=True))
print(f'Deduplicated by {prep.ID_COL}: {before} -> {len(df)} stars')

# Stratified train/val/test split on logAge quantile bins (fixed seed).
rng = np.random.default_rng(SEED)
strata = pd.qcut(df[prep.TARGET], q=10, labels=False, duplicates='drop')
train_idx, val_idx, test_idx = [], [], []
for b in np.unique(strata):
    members = rng.permutation(np.flatnonzero(strata == b))
    n_test = int(round(TEST_FRAC * len(members)))
    n_val = int(round(VAL_FRAC * len(members)))
    test_idx.extend(members[:n_test])
    val_idx.extend(members[n_test:n_test + n_val])
    train_idx.extend(members[n_test + n_val:])
train_raw = df.iloc[sorted(train_idx)].reset_index(drop=True)
val_raw = df.iloc[sorted(val_idx)].reset_index(drop=True)
test_raw = df.iloc[sorted(test_idx)].reset_index(drop=True)
print(f'Train: {len(train_raw)}  |  Val: {len(val_raw)}  |  Test: {len(test_raw)}')
train_raw.head()

## 4. Normalize

Clean labels/errors — **dropping** stars with unphysical (>13.8 Gyr) seismic ages —
then derive `[C/N]` and standardize with **train-only** statistics. Val and test are
transformed with the train statistics; the inverse-frequency age weights are computed
on train only.

In [ ]:
train, train_rep = prep.clean_labels(train_raw, 'train', drop_saturated=True)
val, val_rep = prep.clean_labels(val_raw, 'val', drop_saturated=True)
test, test_rep = prep.clean_labels(test_raw, 'test', drop_saturated=True)

# Dedup-before-split makes overlap impossible; verify rather than assume.
ids = {name: set(s[prep.ID_COL]) for name, s in [('train', train), ('val', val), ('test', test)]}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = ids[a] & ids[b]
    assert not overlap, f'{len(overlap)} stars appear in both {a} and {b}'

# Engineer [C/N] (+ propagated error)
train = prep.derive_features(train)
val = prep.derive_features(val)
test = prep.derive_features(test)

# Fit normalization on TRAIN ONLY, then apply to all splits
stats = prep.fit_norm_stats(train, FEATURES)
train = prep.apply_norm(train, FEATURES, stats)
val = prep.apply_norm(val, FEATURES, stats)
test = prep.apply_norm(test, FEATURES, stats)

# Inverse-frequency sample weights to fight age imbalance (train only)
train, edges, _ = prep.add_sample_weights(train, prep.N_AGE_BINS, WEIGHT_CLIP)

print(f'\nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)} stars')
train[FEATURE_COLS].describe()

In [8]:
prep.validate(train, test, FEATURES, edges)

## 5. Build tensors

In [ ]:
def to_arrays(d):
    X = d[FEATURE_COLS].values.astype(np.float32)
    X_err = d[ERROR_COLS].values.astype(np.float32)
    y = d[prep.TARGET].values.astype(np.float32)
    y_err = d[prep.TARGET_ERR].values.astype(np.float32)
    # Inverse-frequency age weights (mean 1). Val/test have no weights -> ones.
    w = (d['train_weight'].values if 'train_weight' in d.columns
         else np.ones(len(d))).astype(np.float32)
    return X, X_err, y, y_err, w


X_train, X_err_train, y_train, y_err_train, w_train = to_arrays(train)
X_val, X_err_val, y_val, y_err_val, _ = to_arrays(val)
X_test, X_err_test, y_test, y_err_test, _ = to_arrays(test)

print(f'Features: {X_train.shape[1]}  ({FEATURE_COLS})')
print(f'logAge range: [{y_train.min():.3f}, {y_train.max():.3f}]')
print(f'train_weight: mean={w_train.mean():.3f}, min={w_train.min():.3f}, max={w_train.max():.3f}')

X_train_t = torch.FloatTensor(X_train).to(device)
X_err_train_t = torch.FloatTensor(X_err_train).to(device)
y_train_t = torch.FloatTensor(y_train).to(device)
y_err_train_t = torch.FloatTensor(y_err_train).to(device)

# Bare inverse-frequency weights (mean 1). The minibatch-correct ELBO scaling
# (likelihood x N/len(batch)) is applied inside train_smooth_bnn via
# minibatch_scale=True, using the actual batch length each step -- do NOT
# pre-multiply by N/BATCH_SIZE here.
w_train_t = torch.FloatTensor(w_train).to(device)

## 6. Train the network

In [ ]:
set_seed(SEED)

y_mean = float(np.mean(y_train))
y_std = float(np.std(y_train))
print(f'Empirical logAge stats - mean: {y_mean:.3f}, std: {y_std:.3f}')

MODEL_KWARGS = dict(
    input_dim=X_train.shape[1],
    hidden_dim=HIDDEN_DIM,
    use_skip_connections=True,
    use_empirical_output_bias=True,  # mean-zero bias prior: less shrinkage to y_mean
    use_leaky_relu=True,
    y_mean=y_mean,
    y_std=y_std,
)
model = BayesianNeuralNetwork(**MODEL_KWARGS)
model.to(device)

guide, losses = train_smooth_bnn(
    model,
    X_train_t, X_err_train_t,
    y_train_t, y_err_train_t,
    num_iterations=NUM_ITERATIONS,
    initial_lr=INITIAL_LR,
    lr_decay_per_epoch=LR_DECAY_PER_EPOCH,
    batch_size=BATCH_SIZE,
    seed=SEED,
    w_train=w_train_t,        # bare inverse-frequency age weights (mean 1)
    minibatch_scale=True,     # likelihood x N/len(batch), applied internally
)

In [ ]:
import pyro

pyro.get_param_store().save(str(OUTPUT_DIR / 'targeted_bnn_params.pth'))

smoothed = []
for i, loss in enumerate(losses):
    smoothed.append(loss if i == 0 else 0.95 * smoothed[-1] + 0.05 * loss)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses, alpha=0.4, label='Raw loss')
axes[0].plot(smoothed, lw=2, label='Smoothed loss')
axes[0].set_title('Training loss')
axes[0].legend()

# Zoom on the tail: the smoothed curve should be flat by the end -- if it is
# still clearly falling, increase NUM_ITERATIONS.
tail = max(len(losses) - 30, 0)
axes[1].plot(range(tail, len(losses)), losses[tail:], alpha=0.4)
axes[1].plot(range(tail, len(losses)), smoothed[tail:], lw=2)
axes[1].set_title('Last 30 epochs')

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ELBO loss')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Calibrate on val, evaluate on test

Posterior predictions on the **val** split fit a single scalar recalibration factor
for the predicted uncertainties (the std of the z-scores; 1.0 = already calibrated,
< 1 = the raw uncertainties are inflated). The **test** split is then evaluated once,
reporting both raw and calibrated coverage (targets: ~68% within 1σ, ~95% within 2σ).

In [ ]:
def predict_split(X, X_err, y_err, y):
    X_t = torch.FloatTensor(X).to(device)
    X_err_t = torch.FloatTensor(X_err).to(device)
    y_err_t = torch.FloatTensor(y_err).to(device)
    samples, mean_pred, model_unc, intrinsic = get_targeted_posterior_samples(
        model, guide, X_t, X_err_t, y_err_t, num_samples=5000
    )
    return analyze_targeted_results(samples, mean_pred, model_unc, intrinsic, y_err, y)


print('=== VAL (calibration split) ===')
summary_val = predict_split(X_val, X_err_val, y_err_val, y_val)

# Scalar recalibration: if the uncertainties were right, z = residual/sigma
# would have std 1 on val; calib < 1 means the raw sigmas are inflated.
calib = float(summary_val['normalized_residual'].std())
print(f'\nCalibration factor (val z-score std): {calib:.3f}')

print('\n=== TEST (held out, evaluated once) ===')
summary = predict_split(X_test, X_err_test, y_err_test, y_test)
summary['calibrated_uncertainty'] = summary['total_predictive_uncertainty'] * calib
summary['calibrated_z'] = summary['residual'] / summary['calibrated_uncertainty']

print('\nTest coverage (targets: 68% within 1 sigma, 95% within 2):')
for name, z in [('raw       ', summary['normalized_residual']),
                ('calibrated', summary['calibrated_z'])]:
    print(f'  {name}  |z|<1: {(z.abs() < 1).mean():6.1%}   |z|<2: {(z.abs() < 2).mean():6.1%}')

summary.to_csv(OUTPUT_DIR / 'targeted_prediction_summary.csv', index=False)
summary.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(y_test, summary['pred_median'], alpha=0.4, s=10)
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('True logAge')
axes[0].set_ylabel('Predicted logAge')
axes[0].set_title('Predictions vs True')

axes[1].scatter(y_test, summary['residual'], alpha=0.4, s=10)
axes[1].axhline(0, color='r', ls='--')
# Binned median residual: shows where the model is biased vs age
# (e.g. old stars dragged young) rather than just scattered.
bin_ids = np.digitize(y_test, edges[1:-1])
centers, medians = [], []
for b in np.unique(bin_ids):
    m = bin_ids == b
    if m.sum() >= 5:
        centers.append(np.median(y_test[m]))
        medians.append(np.median(summary['residual'].values[m]))
axes[1].plot(centers, medians, 'o-', color='k', lw=2, ms=4, label='binned median')
axes[1].set_xlabel('True logAge')
axes[1].set_ylabel('Residual (True - Predicted)')
axes[1].set_title('Residuals')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from math import erf

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# z-score histogram vs a unit normal
zgrid = np.linspace(-4, 4, 200)
axes[0].hist(summary['normalized_residual'], bins=40, range=(-4, 4), density=True,
             alpha=0.45, label='raw z')
axes[0].hist(summary['calibrated_z'], bins=40, range=(-4, 4), density=True,
             alpha=0.45, label='calibrated z')
axes[0].plot(zgrid, np.exp(-zgrid**2 / 2) / np.sqrt(2 * np.pi), 'k--', label='N(0,1)')
axes[0].set_xlabel('z = residual / predicted sigma')
axes[0].set_ylabel('Density')
axes[0].set_title('Uncertainty calibration (test)')
axes[0].legend()

# Coverage curve: fraction within k*sigma vs the Gaussian expectation
ks = np.linspace(0.1, 3.0, 30)
expected = np.array([erf(k / np.sqrt(2)) for k in ks])
for col, label in [('normalized_residual', 'raw'), ('calibrated_z', 'calibrated')]:
    observed = [(summary[col].abs() < k).mean() for k in ks]
    axes[1].plot(expected, observed, label=label)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('Expected coverage (Gaussian)')
axes[1].set_ylabel('Observed coverage')
axes[1].set_title('Above the diagonal = overestimated sigma')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Data-ceiling check

How close is the BNN to what these features can support? Two plain baselines
(closed-form ridge regression, inverse-distance-weighted kNN) run on the **identical**
split and features.

How to read the numbers:

- If a baseline beats the BNN on correlation/RMS, there is **headroom** — the features
  hold signal the BNN is not extracting, so training/architecture (not the data) is
  the limit.
- Slope < 1 is regression toward the mean. The inverse-frequency age weighting
  deliberately trades some RMS for a slope closer to 1, so the BNN can legitimately
  show worse RMS but a much better slope than an unweighted baseline. Decide which
  matters for the science before "fixing" either.

In [ ]:
def _metrics(y, p):
    return (np.corrcoef(y, p)[0, 1],
            float(np.sqrt(np.mean((y - p) ** 2))),
            float(np.polyfit(y, p, 1)[0]))


# Ridge regression (closed form, small penalty)
A = np.hstack([X_train, np.ones((len(X_train), 1))])
At = np.hstack([X_test, np.ones((len(X_test), 1))])
coef = np.linalg.solve(A.T @ A + 1e-2 * np.eye(A.shape[1]), A.T @ y_train)
ridge_pred = At @ coef

# Inverse-distance-weighted kNN
d = np.sqrt(((X_test[:, None, :] - X_train[None, :, :]) ** 2).sum(-1))
nn = np.argsort(d, axis=1)[:, :10]
dd = np.take_along_axis(d, nn, axis=1)
wts = 1.0 / (dd + 1e-6)
wts /= wts.sum(axis=1, keepdims=True)
knn_pred = (y_train[nn] * wts).sum(axis=1)

results = {name: _metrics(y_test, p)
           for name, p in [('ridge', ridge_pred),
                           ('kNN k=10', knn_pred),
                           ('BNN', summary['pred_median'].values)]}
print(f"{'model':12s} {'corr':>6s} {'RMS':>7s} {'slope':>7s}   (test, n={len(y_test)})")
for name, (c, r, s) in results.items():
    print(f'{name:12s} {c:6.3f} {r:7.3f} {s:7.3f}')
print(f"{'mean-only':12s} {'--':>6s} {y_test.std():7.3f} {0.0:7.3f}   (predict-the-mean floor)")

# Verdict driven by the numbers, not asserted.
bnn_c, bnn_r, bnn_s = results['BNN']
base_c = max(results['ridge'][0], results['kNN k=10'][0])
base_r = min(results['ridge'][1], results['kNN k=10'][1])
print()
if bnn_c >= base_c - 0.01 and bnn_r <= base_r + 0.01:
    print('BNN matches or beats the baselines => it is extracting what the features')
    print('hold; further gains need better labels or features, not more training.')
else:
    print(f'A baseline beats the BNN on point accuracy (corr {base_c:.3f} vs {bnn_c:.3f}, '
          f'RMS {base_r:.3f} vs {bnn_r:.3f}) => headroom remains.')
    print(f'Note the BNN slope ({bnn_s:.3f}) vs the baselines: the age weighting')
    print('deliberately trades RMS for less regression to the mean.')

In [ ]:
import joblib

# Everything needed to reproduce predictions, in one bundle next to the param
# store. NOTE: the learned posterior lives in the Pyro param store
# (targeted_bnn_params.pth, saved in section 6) -- model.state_dict() alone
# cannot restore a Pyro BNN, so no state_dict is saved.
bundle = {
    'features': FEATURES,
    'stats': stats,                 # per-feature {"mean","std"} from TRAIN only
    'y_mean': y_mean,
    'y_std': y_std,
    'model_kwargs': MODEL_KWARGS,   # re-instantiate the identical architecture
    'age_bin_edges': edges,
    'weight_clip': WEIGHT_CLIP,
    'seed': SEED,
    'calibration_factor': calib,    # multiply predicted sigmas by this
    'param_store_file': 'targeted_bnn_params.pth',
}
joblib.dump(bundle, OUTPUT_DIR / 'bnn_inference_bundle_rgb.pkl')
print(f"Saved {OUTPUT_DIR / 'bnn_inference_bundle_rgb.pkl'}")

# Reload recipe:
#   b = joblib.load(OUTPUT_DIR / 'bnn_inference_bundle_rgb.pkl')
#   model = BayesianNeuralNetwork(**b['model_kwargs']).to(device)
#   pyro.clear_param_store()
#   pyro.get_param_store().load(str(OUTPUT_DIR / b['param_store_file']), map_location=device)
#   guide = AutoDiagonalNormal(model)  # picks up the loaded params by name on first use
# then normalize inputs with prep.apply_norm(df, b['features'], b['stats']) and call
# get_targeted_posterior_samples(model, guide, ...); multiply the returned sigmas by
# b['calibration_factor'].